In [0]:
#Display the contents of the Region.csv file
display(spark.read.csv("/Volumes/workspace/default/adventure_works_volume/bronze/Region.csv", header=True, sep="\t"))

In [0]:
#Define the Schema for Region 
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

region_schema = StructType([
    StructField("SalesTerritoryKey", IntegerType(),True),
    StructField("Region", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Group", StringType(), True)
])

#Create DataFrame for Region using predefined schema
df_region = spark.read.csv("/Volumes/workspace/default/adventure_works_volume/bronze/Region.csv",  header=True, sep="\t", schema=region_schema)


In [0]:
#Rename Column SalesTerritoryKey to Sales_Territory_Key
from pyspark.sql import functions as F

df_region = df_region.withColumnRenamed("SalesTerritoryKey", "Sales_Territory_Key")

In [0]:
#Trim Spaces
df_region = df_region.select(
    *(F.trim(F.col(c)).alias(c) 
      if isinstance(c, StringType) #if column is StringType then trim spaces, else keep it unchanged
      else F.col(c) for c in df_region.columns)
)

In [0]:
#Drop Duplicates
df_region = df_region.dropDuplicates()
display(df_region)

In [0]:
#Drop rows with missing Sales_Territory_Key
df_region = df_region.dropna(subset=["Sales_Territory_Key"])

#Fill missing values in Region, Country, Group columns with 'Unknown' 
df_region = df_region.fillna('Unknown',subset=["Region","Country","Group"])
display(df_region)


In [0]:
df_region.select("Country").distinct().display()

In [0]:
#Replace United States with USA for standardization of Country column
df_region = df_region.replace({"United States" : "USA"}, subset=["Country"])
display(df_region)

In [0]:
df_region.select("Country").distinct().display()

In [0]:
#Save Region DataFrame as Delta Table
df_region.write.mode("overwrite").format("delta").saveAsTable("adventure_works_db.region_delta")
display(spark.sql("show tables in adventure_works_db"))
